In [0]:
# Purpose: Configure the incremental pipeline and inspect the Bronze Delta table.
# Safety: Read-only inspection; does not modify data or checkpoints.

from pyspark.sql import functions as F
from delta.tables import DeltaTable

bronze_path = (
    "abfss://trackpulse-data@trackpulsestorage.dfs.core.windows.net/"
    "bronze/telemetry_stream/"
)

silver_path = (
    "abfss://trackpulse-data@trackpulsestorage.dfs.core.windows.net/"
    "silver/telemetry_stream/"
)

quarantine_path = (
    "abfss://trackpulse-data@trackpulsestorage.dfs.core.windows.net/"
    "silver/quarantine/telemetry_stream/"
)

cdf_checkpoint_path = (
    "abfss://trackpulse-data@trackpulsestorage.dfs.core.windows.net/"
    "checkpoints/databricks_streaming/silver_cdf/"
)

bronze_df = spark.read.format("delta").load(bronze_path)

print(f"Bronze records: {bronze_df.count():,}")

print("=== Recent Bronze Delta history ===")
display(
    DeltaTable.forPath(spark, bronze_path)
    .history(5)
    .select("version", "timestamp", "operation")
)

Bronze records: 41,522
=== Recent Bronze Delta history ===


version,timestamp,operation
4,2026-10-28T17:56:19.000Z,SET TBLPROPERTIES
3,2026-10-28T15:20:51.000Z,SET TBLPROPERTIES
2,2026-10-28T13:36:56.000Z,STREAMING UPDATE
1,2026-10-28T13:36:54.000Z,STREAMING UPDATE
0,2026-10-28T13:36:52.000Z,STREAMING UPDATE


In [0]:
# Purpose: Inspect the Bronze Delta table properties and current CDF configuration.
# Safety: Read-only metadata inspection; does not modify data or table properties.

detail_df = spark.sql(f"DESCRIBE DETAIL delta.`{bronze_path}`")
display(
    detail_df.select(
        "format",
        "location",
        "properties"
    )
)

format,location,properties
delta,abfss://trackpulse-data@trackpulsestorage.dfs.core.windows.net/bronze/telemetry_stream,Map(delta.enableChangeDataFeed -> true)


In [0]:
# Purpose: Enable Change Data Feed for future Bronze Delta table changes.
# Safety: Changes a Delta table property only; does not rewrite existing records.
# Historical changes before CDF activation will not be available through CDF.

spark.sql(f"""
    ALTER TABLE delta.`{bronze_path}`
    SET TBLPROPERTIES (
        delta.enableChangeDataFeed = true
    )
""")

print("Change Data Feed enabled for future Bronze changes.")

Change Data Feed enabled for future Bronze changes.


In [0]:
# Purpose: Verify that Change Data Feed is enabled.
# Safety: Read-only metadata check.

detail_df = spark.sql(f"DESCRIBE DETAIL delta.`{bronze_path}`")

display(detail_df.select("properties"))

properties
Map(delta.enableChangeDataFeed -> true)


In [0]:
# Purpose: Identify the current Bronze Delta version after enabling CDF.
# Safety: Read-only history inspection; does not modify data or checkpoints.

from delta.tables import DeltaTable

bronze_history = (
    DeltaTable.forPath(spark, bronze_path)
    .history(10)
)

display(
    bronze_history.select(
        "version",
        "timestamp",
        "operation",
        "operationParameters"
    )
)

version,timestamp,operation,operationParameters
5,2026-10-28T18:02:17.000Z,SET TBLPROPERTIES,"Map(properties -> {""delta.enableChangeDataFeed"":""true""})"
4,2026-10-28T17:56:19.000Z,SET TBLPROPERTIES,"Map(properties -> {""delta.enableChangeDataFeed"":""true""})"
3,2026-10-28T15:20:51.000Z,SET TBLPROPERTIES,"Map(properties -> {""delta.enableChangeDataFeed"":""true""})"
2,2026-10-28T13:36:56.000Z,STREAMING UPDATE,"Map(outputMode -> Append, queryId -> 348c575c-d0f7-4ec6-a64a-1208d3a1f61c, epochId -> 1, statsOnLoad -> false)"
1,2026-10-28T13:36:54.000Z,STREAMING UPDATE,"Map(outputMode -> Append, queryId -> 348c575c-d0f7-4ec6-a64a-1208d3a1f61c, epochId -> 0, statsOnLoad -> false)"
0,2026-10-28T13:36:52.000Z,STREAMING UPDATE,"Map(outputMode -> Append, queryId -> 348c575c-d0f7-4ec6-a64a-1208d3a1f61c, epochId -> -1, statsOnLoad -> false)"


In [0]:
# Purpose: Test Change Data Feed using the Bronze Delta storage path.
# Safety: Read-only test; does not write to Bronze or Silver.

from delta.tables import DeltaTable

history_df = DeltaTable.forPath(spark, bronze_path).history(1)
latest_version = history_df.select("version").first()["version"]

print(f"Latest Bronze version: {latest_version}")

try:
    cdf_test_df = (
        spark.read.format("delta")
        .option("readChangeFeed", "true")
        .option("startingVersion", latest_version)
        .load(bronze_path)
    )

    print("CDF read initialized successfully.")
    cdf_test_df.printSchema()

    display(cdf_test_df.limit(5))

except Exception as e:
    print("CDF validation error:")
    print(str(e)[:1500])

Latest Bronze version: 5
CDF read initialized successfully.
root
 |-- capture_sequence_number: long (nullable = true)
 |-- capture_enqueued_time_utc: string (nullable = true)
 |-- event_id: string (nullable = true)
 |-- event_timestamp: string (nullable = true)
 |-- vehicle_id: string (nullable = true)
 |-- latitude: double (nullable = true)
 |-- longitude: double (nullable = true)
 |-- speed_kph: double (nullable = true)
 |-- odometer_km: double (nullable = true)
 |-- rpm: long (nullable = true)
 |-- fuel_rate_lph: double (nullable = true)
 |-- engine_temp_c: double (nullable = true)
 |-- ignition_on: boolean (nullable = true)
 |-- trip_id: string (nullable = true)
 |-- shipment_id: string (nullable = true)
 |-- driver_id: string (nullable = true)
 |-- route_id: string (nullable = true)
 |-- cargo_temp_c: double (nullable = true)
 |-- event_json: string (nullable = true)
 |-- event_timestamp_utc: timestamp (nullable = true)
 |-- bronze_ingested_at: timestamp (nullable = true)
 |-- sou

capture_sequence_number,capture_enqueued_time_utc,event_id,event_timestamp,vehicle_id,latitude,longitude,speed_kph,odometer_km,rpm,fuel_rate_lph,engine_temp_c,ignition_on,trip_id,shipment_id,driver_id,route_id,cargo_temp_c,event_json,event_timestamp_utc,bronze_ingested_at,source_format,_change_type,_commit_version,_commit_timestamp


In [0]:
# Purpose: Inspect available CDF changes and their commit versions.
# Safety: Read-only test; does not modify Bronze, Silver, or checkpoints.

cdf_summary_df = (
    cdf_test_df
    .groupBy("_change_type", "_commit_version")
    .count()
    .orderBy("_commit_version", "_change_type")
)

display(cdf_summary_df)

print(f"CDF records returned: {cdf_test_df.count():,}")

_change_type,_commit_version,count


CDF records returned: 0


In [0]:
# Purpose: Check the latest Bronze Delta version and record count before an incremental test.
# Safety: Read-only inspection; does not modify data or checkpoints.

from delta.tables import DeltaTable

latest_history = (
    DeltaTable.forPath(spark, bronze_path)
    .history(5)
    .select("version", "timestamp", "operation")
)

display(latest_history)

print(
    "Current Bronze count:",
    spark.read.format("delta").load(bronze_path).count()
)

version,timestamp,operation
5,2026-10-28T18:02:17.000Z,SET TBLPROPERTIES
4,2026-10-28T17:56:19.000Z,SET TBLPROPERTIES
3,2026-10-28T15:20:51.000Z,SET TBLPROPERTIES
2,2026-10-28T13:36:56.000Z,STREAMING UPDATE
1,2026-10-28T13:36:54.000Z,STREAMING UPDATE


Current Bronze count: 41522


In [0]:
# Purpose: Check the latest Bronze Delta version and summarize its recent history.
# Safety: Read-only; does not modify data, tables, or checkpoints.

from delta.tables import DeltaTable

display(
    DeltaTable.forPath(spark, bronze_path)
    .history(5)
    .select("version", "timestamp", "operation")
)

print(
    "Current Bronze records:",
    spark.read.format("delta").load(bronze_path).count()
)

version,timestamp,operation
5,2026-10-28T18:02:17.000Z,SET TBLPROPERTIES
4,2026-10-28T17:56:19.000Z,SET TBLPROPERTIES
3,2026-10-28T15:20:51.000Z,SET TBLPROPERTIES
2,2026-10-28T13:36:56.000Z,STREAMING UPDATE
1,2026-10-28T13:36:54.000Z,STREAMING UPDATE


Current Bronze records: 41522


In [0]:
# Inspect the existing Silver and Quarantine schemas without modifying data.

for table_name, table_path in [
    ("Silver", silver_path),
    ("Quarantine", quarantine_path),
]:
    print(f"\n--- {table_name} ---")

    try:
        df = spark.read.format("delta").load(table_path)
        print("Columns:", df.columns)
        df.printSchema()
    except Exception as e:
        print(f"Could not inspect {table_name}: {str(e)[:500]}")


--- Silver ---
Columns: ['event_id', 'event_timestamp', 'vehicle_id', 'latitude', 'longitude', 'speed_kph', 'odometer_km', 'rpm', 'fuel_rate_lph', 'engine_temp_c', 'ignition_on', 'trip_id', 'shipment_id', 'driver_id', 'route_id', 'cargo_temp_c', 'ingested_at', 'source_file']
root
 |-- event_id: string (nullable = true)
 |-- event_timestamp: timestamp (nullable = true)
 |-- vehicle_id: string (nullable = true)
 |-- latitude: double (nullable = true)
 |-- longitude: double (nullable = true)
 |-- speed_kph: double (nullable = true)
 |-- odometer_km: double (nullable = true)
 |-- rpm: long (nullable = true)
 |-- fuel_rate_lph: double (nullable = true)
 |-- engine_temp_c: double (nullable = true)
 |-- ignition_on: boolean (nullable = true)
 |-- trip_id: string (nullable = true)
 |-- shipment_id: string (nullable = true)
 |-- driver_id: string (nullable = true)
 |-- route_id: string (nullable = true)
 |-- cargo_temp_c: double (nullable = true)
 |-- ingested_at: timestamp (nullable = true)
 

In [0]:

# Purpose: Process available Bronze events incrementally into Silver and Quarantine.
# Safety: Never overwrite historical Silver, delete tables, or reset checkpoints.
# Idempotency: Use stable keys and update existing records only when values change.

from pyspark.sql import functions as F
from pyspark.sql.window import Window
from delta.tables import DeltaTable

# ------------------------------------------------------------
# 1. Define the existing storage paths
# ------------------------------------------------------------

bronze_path = (
    "abfss://trackpulse-data@trackpulsestorage.dfs.core.windows.net/"
    "bronze/telemetry_stream/"
)

silver_path = (
    "abfss://trackpulse-data@trackpulsestorage.dfs.core.windows.net/"
    "silver/telemetry_stream/"
)

quarantine_path = (
    "abfss://trackpulse-data@trackpulsestorage.dfs.core.windows.net/"
    "silver/quarantine/telemetry_stream/"
)

# Reuse the existing checkpoint. Do not delete or reset it.
silver_checkpoint_path = (
    "abfss://trackpulse-data@trackpulsestorage.dfs.core.windows.net/"
    "checkpoints/databricks_streaming/silver_incremental_v1/"
)

# ------------------------------------------------------------
# 2. Validate the existing Delta tables
# ------------------------------------------------------------

bronze_df = spark.read.format("delta").load(bronze_path)
silver_df = spark.read.format("delta").load(silver_path)
quarantine_df = spark.read.format("delta").load(quarantine_path)

required_bronze_columns = [
    "event_id",
    "event_timestamp",
    "vehicle_id",
    "latitude",
    "longitude",
    "speed_kph",
    "odometer_km",
    "rpm",
    "fuel_rate_lph",
    "engine_temp_c",
    "ignition_on",
    "trip_id",
    "shipment_id",
    "driver_id",
    "route_id",
    "cargo_temp_c",
    "event_timestamp_utc",
    "bronze_ingested_at",
    "source_format",
    "event_json",
    "capture_sequence_number",
    "capture_enqueued_time_utc",
]

missing_columns = [
    c for c in required_bronze_columns
    if c not in bronze_df.columns
]

if missing_columns:
    raise ValueError(
        f"Bronze schema is missing required columns: {missing_columns}"
    )

expected_silver_columns = [
    "event_id",
    "event_timestamp",
    "vehicle_id",
    "latitude",
    "longitude",
    "speed_kph",
    "odometer_km",
    "rpm",
    "fuel_rate_lph",
    "engine_temp_c",
    "ignition_on",
    "trip_id",
    "shipment_id",
    "driver_id",
    "route_id",
    "cargo_temp_c",
    "ingested_at",
    "source_file",
]

if set(silver_df.columns) != set(expected_silver_columns):
    raise ValueError(
        "Silver schema differs from the expected schema. "
        "Inspect it before running writes."
    )

print("Bronze schema validated.")
print("Silver schema validated.")
print("Existing Bronze records:", bronze_df.count())
print("Existing Silver records:", silver_df.count())
print("Existing Quarantine records:", quarantine_df.count())

# ------------------------------------------------------------
# 3. Add missing Quarantine metadata columns if necessary
# ------------------------------------------------------------

if "quarantine_key" not in quarantine_df.columns:
    spark.sql(f"""
        ALTER TABLE delta.`{quarantine_path}`
        ADD COLUMNS (quarantine_key STRING)
    """)

if "dq_reasons" not in quarantine_df.columns:
    spark.sql(f"""
        ALTER TABLE delta.`{quarantine_path}`
        ADD COLUMNS (dq_reasons STRING)
    """)

# ------------------------------------------------------------
# 4. Backfill only missing Quarantine keys
# ------------------------------------------------------------

# Keep existing keys unchanged.
# Use the same fallback payload fields used by the streaming function.

spark.sql(f"""
    UPDATE delta.`{quarantine_path}`
    SET quarantine_key =
        CASE
            WHEN event_id IS NOT NULL AND trim(event_id) != ''
                THEN concat('event:', event_id)
            ELSE concat(
                'payload:',
                sha2(
                    coalesce(
                        event_json,
                        to_json(named_struct(
                            'capture_sequence_number',
                                capture_sequence_number,
                            'capture_enqueued_time_utc',
                                capture_enqueued_time_utc,
                            'event_id', event_id,
                            'event_timestamp', event_timestamp,
                            'vehicle_id', vehicle_id,
                            'latitude', latitude,
                            'longitude', longitude,
                            'speed_kph', speed_kph,
                            'odometer_km', odometer_km,
                            'rpm', rpm,
                            'fuel_rate_lph', fuel_rate_lph,
                            'engine_temp_c', engine_temp_c,
                            'ignition_on', ignition_on,
                            'trip_id', trip_id,
                            'shipment_id', shipment_id,
                            'driver_id', driver_id,
                            'route_id', route_id,
                            'cargo_temp_c', cargo_temp_c
                        ))
                    ),
                    256
                )
            )
        END
    WHERE quarantine_key IS NULL OR trim(quarantine_key) = ''
""")

# ------------------------------------------------------------
# 5. Define the incremental batch-processing function
# ------------------------------------------------------------

def process_silver_batch(batch_df, batch_id):

    # Skip empty batches.
    if batch_df.limit(1).count() == 0:
        print(f"Batch {batch_id}: empty batch; nothing to process.")
        return

    # Parse timestamps and retain ingestion metadata.
    prepared_df = (
        batch_df
        .withColumn(
            "event_timestamp_parsed",
            F.coalesce(
                F.to_timestamp(F.col("event_timestamp")),
                F.col("event_timestamp_utc")
            )
        )
        .withColumn(
            "ingested_at",
            F.coalesce(
                F.col("bronze_ingested_at"),
                F.current_timestamp()
            )
        )
    )

    # Evaluate data-quality rules.
    prepared_df = prepared_df.withColumn(
        "dq_reasons",
        F.concat_ws(
            "; ",
            F.when(
                F.col("event_id").isNull()
                | (F.trim(F.col("event_id")) == ""),
                F.lit("missing_event_id")
            ),
            F.when(
                F.col("event_timestamp_parsed").isNull(),
                F.lit("invalid_event_timestamp")
            ),
            F.when(
                F.col("latitude").isNull()
                | (F.col("latitude") < -90)
                | (F.col("latitude") > 90),
                F.lit("invalid_latitude")
            ),
            F.when(
                F.col("longitude").isNull()
                | (F.col("longitude") < -180)
                | (F.col("longitude") > 180),
                F.lit("invalid_longitude")
            ),
            F.when(
                F.col("speed_kph") < 0,
                F.lit("negative_speed")
            ),
            F.when(
                F.col("odometer_km") < 0,
                F.lit("negative_odometer")
            ),
            F.when(
                F.col("rpm") < 0,
                F.lit("negative_rpm")
            ),
            F.when(
                F.col("fuel_rate_lph") < 0,
                F.lit("negative_fuel_rate")
            )
        )
    )

    # Keep valid records only for Silver.
    valid_df = prepared_df.filter(F.col("dq_reasons") == "")

    # Keep one deterministic source row per event_id in this batch.
    valid_window = Window.partitionBy("event_id").orderBy(
        F.col("bronze_ingested_at").desc_nulls_last(),
        F.col("capture_sequence_number").desc_nulls_last()
    )

    valid_df = (
        valid_df
        .withColumn("_row_number", F.row_number().over(valid_window))
        .filter(F.col("_row_number") == 1)
        .select(
            "event_id",
            F.col("event_timestamp_parsed").alias("event_timestamp"),
            "vehicle_id",
            "latitude",
            "longitude",
            "speed_kph",
            "odometer_km",
            "rpm",
            "fuel_rate_lph",
            "engine_temp_c",
            "ignition_on",
            "trip_id",
            "shipment_id",
            "driver_id",
            "route_id",
            "cargo_temp_c",
            "ingested_at",
            F.lit(None).cast("string").alias("source_file")
        )
    )

    # Update Silver only when an existing event has changed.
    silver_compare_columns = [
        "event_timestamp",
        "vehicle_id",
        "latitude",
        "longitude",
        "speed_kph",
        "odometer_km",
        "rpm",
        "fuel_rate_lph",
        "engine_temp_c",
        "ignition_on",
        "trip_id",
        "shipment_id",
        "driver_id",
        "route_id",
        "cargo_temp_c",
        "ingested_at",
        "source_file",
    ]

    silver_changed_condition = "NOT (" + " AND ".join(
        f"t.`{c}` <=> s.`{c}`"
        for c in silver_compare_columns
    ) + ")"

    if valid_df.limit(1).count() > 0:
        (
            DeltaTable.forPath(spark, silver_path)
            .alias("t")
            .merge(
                valid_df.alias("s"),
                "t.event_id = s.event_id"
            )
            .whenMatchedUpdateAll(
                condition=silver_changed_condition
            )
            .whenNotMatchedInsertAll()
            .execute()
        )

    # --------------------------------------------------------
    # Prepare invalid records for Quarantine
    # --------------------------------------------------------

    # Use a deterministic key for records without event IDs.
    key_payload_columns = [
        "capture_sequence_number",
        "capture_enqueued_time_utc",
        "event_id",
        "event_timestamp",
        "vehicle_id",
        "latitude",
        "longitude",
        "speed_kph",
        "odometer_km",
        "rpm",
        "fuel_rate_lph",
        "engine_temp_c",
        "ignition_on",
        "trip_id",
        "shipment_id",
        "driver_id",
        "route_id",
        "cargo_temp_c",
    ]

    payload_json = F.to_json(
        F.struct(*[F.col(c) for c in key_payload_columns])
    )

    quarantine_df_batch = (
        prepared_df
        .filter(F.col("dq_reasons") != "")
        .withColumn(
            "quarantine_key",
            F.when(
                F.col("event_id").isNotNull()
                & (F.trim(F.col("event_id")) != ""),
                F.concat(F.lit("event:"), F.col("event_id"))
            ).otherwise(
                F.concat(
                    F.lit("payload:"),
                    F.sha2(
                        F.coalesce(F.col("event_json"), payload_json),
                        256
                    )
                )
            )
        )
    )

    # Keep one row per quarantine key within this batch.
    quarantine_window = Window.partitionBy("quarantine_key").orderBy(
        F.col("bronze_ingested_at").desc_nulls_last(),
        F.col("capture_sequence_number").desc_nulls_last()
    )

    quarantine_df_batch = (
        quarantine_df_batch
        .withColumn("_row_number", F.row_number().over(quarantine_window))
        .filter(F.col("_row_number") == 1)
        .drop("_row_number", "event_timestamp_parsed", "ingested_at")
    )

    # Align the source columns with the actual Quarantine schema.
    quarantine_target = spark.read.format("delta").load(quarantine_path)
    target_columns = quarantine_target.columns

    for column_name in target_columns:
        if column_name not in quarantine_df_batch.columns:
            quarantine_df_batch = quarantine_df_batch.withColumn(
                column_name,
                F.lit(None).cast(
                    quarantine_target.schema[column_name].dataType
                )
            )

    quarantine_df_batch = quarantine_df_batch.select(*target_columns)

    # Update Quarantine only when a stored value has changed.
    quarantine_compare_columns = [
        c for c in target_columns
        if c != "quarantine_key"
    ]

    quarantine_changed_condition = "NOT (" + " AND ".join(
        f"t.`{c}` <=> s.`{c}`"
        for c in quarantine_compare_columns
    ) + ")"

    if quarantine_df_batch.limit(1).count() > 0:
        (
            DeltaTable.forPath(spark, quarantine_path)
            .alias("t")
            .merge(
                quarantine_df_batch.alias("s"),
                "t.quarantine_key = s.quarantine_key"
            )
            .whenMatchedUpdateAll(
                condition=quarantine_changed_condition
            )
            .whenNotMatchedInsertAll()
            .execute()
        )

    print(f"Batch {batch_id} processed successfully.")

# ------------------------------------------------------------
# 6. Process available Bronze changes using the existing checkpoint
# ------------------------------------------------------------

if len(spark.streams.active) > 0:
    raise RuntimeError(
        "An active streaming query exists. Stop and inspect it "
        "before starting another query."
    )

bronze_stream = (
    spark.readStream
    .format("delta")
    .load(bronze_path)
)

silver_query = (
    bronze_stream.writeStream
    .foreachBatch(process_silver_batch)
    .option("checkpointLocation", silver_checkpoint_path)
    .trigger(availableNow=True)
    .start()
)

silver_query.awaitTermination()

print("Incremental Bronze-to-Silver processing finished.")


Bronze schema validated.
Silver schema validated.
Existing Bronze records: 167522
Existing Silver records: 41510
Existing Quarantine records: 12
Incremental Bronze-to-Silver processing finished.


In [0]:

# Purpose: Validate Silver and Quarantine after incremental processing.
# Safety: Read-only counts and duplicate checks.

silver_check = spark.read.format("delta").load(silver_path)
quarantine_check = spark.read.format("delta").load(quarantine_path)

print("Silver rows:", silver_check.count())
print("Silver distinct event IDs:", 
      silver_check.select("event_id").distinct().count())

print("Quarantine rows:", quarantine_check.count())
print("Quarantine distinct keys:",
      quarantine_check.select("quarantine_key").distinct().count())

print("Null Silver event IDs:",
      silver_check.filter(F.col("event_id").isNull()).count())

print("Null Quarantine keys:",
      quarantine_check.filter(F.col("quarantine_key").isNull()).count())

display(
    quarantine_check
    .groupBy("dq_reasons")
    .count()
    .orderBy(F.desc("count"))
)


Silver rows: 166010
Silver distinct event IDs: 166010
Quarantine rows: 12
Quarantine distinct keys: 12
Null Silver event IDs: 0
Null Quarantine keys: 0


dq_reasons,count
invalid_event_timestamp; invalid_latitude; invalid_longitude,12


In [0]:
# Inspect recent Delta Lake operations to verify incremental behavior

silver_path = (
    "abfss://trackpulse-data@trackpulsestorage.dfs.core.windows.net/"
    "silver/telemetry_stream/"
)

display(
    spark.sql(f"""
        DESCRIBE HISTORY delta.`{silver_path}`
    """).select(
        "version",
        "timestamp",
        "operation",
        "operationMetrics"
    ).limit(10)
)

version,timestamp,operation,operationMetrics
3,2026-10-28T19:38:36.000Z,MERGE,"Map(numTargetRowsCopied -> 0, numTargetRowsDeleted -> 0, numTargetFilesAdded -> 7, numTargetBytesAdded -> 5879986, numTargetBytesRemoved -> 0, numTargetDeletionVectorsAdded -> 2, numTargetRowsMatchedUpdated -> 1500, executionTimeMs -> 3589, materializeSourceTimeMs -> 870, numTargetRowsInserted -> 124500, numTargetRowsMatchedDeleted -> 0, numTargetDeletionVectorsUpdated -> 0, scanTimeMs -> 1019, numTargetRowsUpdated -> 1500, numOutputRows -> 126000, numTargetDeletionVectorsRemoved -> 0, numTargetRowsNotMatchedBySourceUpdated -> 0, numTargetChangeFilesAdded -> 0, numSourceRows -> 126000, numTargetFilesRemoved -> 0, numTargetRowsNotMatchedBySourceDeleted -> 0, rewriteTimeMs -> 1672)"
2,2026-10-28T18:08:01.000Z,MERGE,"Map(numTargetRowsCopied -> 0, numTargetRowsDeleted -> 0, numTargetFilesAdded -> 2, numTargetBytesAdded -> 1849532, numTargetBytesRemoved -> 1894236, numTargetDeletionVectorsAdded -> 0, numTargetRowsMatchedUpdated -> 41510, executionTimeMs -> 3736, materializeSourceTimeMs -> 924, numTargetRowsInserted -> 0, numTargetRowsMatchedDeleted -> 0, numTargetDeletionVectorsUpdated -> 0, scanTimeMs -> 1276, numTargetRowsUpdated -> 41510, numOutputRows -> 41510, numTargetDeletionVectorsRemoved -> 0, numTargetRowsNotMatchedBySourceUpdated -> 0, numTargetChangeFilesAdded -> 0, numSourceRows -> 41510, numTargetFilesRemoved -> 1, numTargetRowsNotMatchedBySourceDeleted -> 0, rewriteTimeMs -> 1490)"
1,2026-10-28T18:03:03.000Z,MERGE,"Map(numTargetRowsCopied -> 0, numTargetRowsDeleted -> 0, numTargetFilesAdded -> 0, numTargetBytesAdded -> 0, numTargetBytesRemoved -> 0, numTargetDeletionVectorsAdded -> 0, numTargetRowsMatchedUpdated -> 0, executionTimeMs -> 1831, materializeSourceTimeMs -> 17, numTargetRowsInserted -> 0, numTargetRowsMatchedDeleted -> 0, numTargetDeletionVectorsUpdated -> 0, scanTimeMs -> 0, numTargetRowsUpdated -> 0, numOutputRows -> 0, numTargetDeletionVectorsRemoved -> 0, numTargetRowsNotMatchedBySourceUpdated -> 0, numTargetChangeFilesAdded -> 0, numSourceRows -> 41510, numTargetFilesRemoved -> 0, numTargetRowsNotMatchedBySourceDeleted -> 0, rewriteTimeMs -> 1753)"
0,2026-10-28T14:30:11.000Z,WRITE,"Map(numFiles -> 1, numOutputRows -> 41510, numOutputBytes -> 1894236)"


In [0]:
# Check the latest committed Delta version and timestamp

silver_path = (
    "abfss://trackpulse-data@trackpulsestorage.dfs.core.windows.net/"
    "silver/telemetry_stream/"
)

history_df = spark.sql(
    f"DESCRIBE HISTORY delta.`{silver_path}`"
)

display(
    history_df
    .select("version", "timestamp", "operation")
    .orderBy("version", ascending=False)
    .limit(5)
)

version,timestamp,operation
2,2026-10-28T18:08:01.000Z,MERGE
1,2026-10-28T18:03:03.000Z,MERGE
0,2026-10-28T14:30:11.000Z,WRITE


In [0]:
# Inspect active streaming queries and their latest progress

for query in spark.streams.active:
    print("Query ID:", query.id)
    print("Query name:", query.name)
    print("Is active:", query.isActive)
    print("Status:", query.status)
    print("Last progress:", query.lastProgress)
    print("Exception:", query.exception())

In [0]:
# Inspect the existing checkpoint without modifying it

checkpoint_path = (
    "abfss://trackpulse-data@trackpulsestorage.dfs.core.windows.net/"
    "checkpoints/databricks_streaming/silver_incremental_v1/"
)

display(
    dbutils.fs.ls(checkpoint_path)
)

print("Offsets directory:")
display(
    dbutils.fs.ls(checkpoint_path + "offsets/")
)

print("Commits directory:")
display(
    dbutils.fs.ls(checkpoint_path + "commits/")
)

path,name,size,modificationTime
abfss://trackpulse-data@trackpulsestorage.dfs.core.windows.net/checkpoints/databricks_streaming/silver_incremental_v1/__tmp_path_dir/,__tmp_path_dir/,0,1793210573000
abfss://trackpulse-data@trackpulsestorage.dfs.core.windows.net/checkpoints/databricks_streaming/silver_incremental_v1/commits/,commits/,0,1793210573000
abfss://trackpulse-data@trackpulsestorage.dfs.core.windows.net/checkpoints/databricks_streaming/silver_incremental_v1/metadata,metadata,45,1793210573000
abfss://trackpulse-data@trackpulsestorage.dfs.core.windows.net/checkpoints/databricks_streaming/silver_incremental_v1/offsets/,offsets/,0,1793210573000


Offsets directory:


path,name,size,modificationTime
abfss://trackpulse-data@trackpulsestorage.dfs.core.windows.net/checkpoints/databricks_streaming/silver_incremental_v1/offsets/0,0,2591,1793210577000
abfss://trackpulse-data@trackpulsestorage.dfs.core.windows.net/checkpoints/databricks_streaming/silver_incremental_v1/offsets/__tmp_path_dir/,__tmp_path_dir/,0,1793210576000


Commits directory:


path,name,size,modificationTime
abfss://trackpulse-data@trackpulsestorage.dfs.core.windows.net/checkpoints/databricks_streaming/silver_incremental_v1/commits/0,0,29,1793210885000
abfss://trackpulse-data@trackpulsestorage.dfs.core.windows.net/checkpoints/databricks_streaming/silver_incremental_v1/commits/__tmp_path_dir/,__tmp_path_dir/,0,1793210885000


In [0]:
# Read-only inspection of recent Bronze and Silver Delta commits
# This cell does not write data or change checkpoints.

bronze_path = (
    "abfss://trackpulse-data@trackpulsestorage.dfs.core.windows.net/"
    "bronze/telemetry_stream/"
)

silver_path = (
    "abfss://trackpulse-data@trackpulsestorage.dfs.core.windows.net/"
    "silver/telemetry_stream/"
)

print("===== BRONZE HISTORY =====")
display(
    spark.sql(f"DESCRIBE HISTORY delta.`{bronze_path}`")
    .select("version", "timestamp", "operation", "operationMetrics")
    .orderBy("version", ascending=False)
    .limit(10)
)

print("===== SILVER HISTORY =====")
display(
    spark.sql(f"DESCRIBE HISTORY delta.`{silver_path}`")
    .select("version", "timestamp", "operation", "operationMetrics")
    .orderBy("version", ascending=False)
    .limit(10)
)

===== BRONZE HISTORY =====


version,timestamp,operation,operationMetrics
5,2026-10-28T18:02:17.000Z,SET TBLPROPERTIES,Map()
4,2026-10-28T17:56:19.000Z,SET TBLPROPERTIES,Map()
3,2026-10-28T15:20:51.000Z,SET TBLPROPERTIES,Map()
2,2026-10-28T13:36:56.000Z,STREAMING UPDATE,"Map(numRemovedFiles -> 0, conflictDetectionTimeMs -> 42, numOutputRows -> 10500, numOutputBytes -> 1563629, numAddedFiles -> 1)"
1,2026-10-28T13:36:54.000Z,STREAMING UPDATE,"Map(numRemovedFiles -> 0, numOutputRows -> 31022, numOutputBytes -> 4451076, numAddedFiles -> 1)"
0,2026-10-28T13:36:52.000Z,STREAMING UPDATE,Map()


===== SILVER HISTORY =====


version,timestamp,operation,operationMetrics
2,2026-10-28T18:08:01.000Z,MERGE,"Map(numTargetRowsCopied -> 0, numTargetRowsDeleted -> 0, numTargetFilesAdded -> 2, numTargetBytesAdded -> 1849532, numTargetBytesRemoved -> 1894236, numTargetDeletionVectorsAdded -> 0, numTargetRowsMatchedUpdated -> 41510, executionTimeMs -> 3736, materializeSourceTimeMs -> 924, numTargetRowsInserted -> 0, numTargetRowsMatchedDeleted -> 0, numTargetDeletionVectorsUpdated -> 0, scanTimeMs -> 1276, numTargetRowsUpdated -> 41510, numOutputRows -> 41510, numTargetDeletionVectorsRemoved -> 0, numTargetRowsNotMatchedBySourceUpdated -> 0, numTargetChangeFilesAdded -> 0, numSourceRows -> 41510, numTargetFilesRemoved -> 1, numTargetRowsNotMatchedBySourceDeleted -> 0, rewriteTimeMs -> 1490)"
1,2026-10-28T18:03:03.000Z,MERGE,"Map(numTargetRowsCopied -> 0, numTargetRowsDeleted -> 0, numTargetFilesAdded -> 0, numTargetBytesAdded -> 0, numTargetBytesRemoved -> 0, numTargetDeletionVectorsAdded -> 0, numTargetRowsMatchedUpdated -> 0, executionTimeMs -> 1831, materializeSourceTimeMs -> 17, numTargetRowsInserted -> 0, numTargetRowsMatchedDeleted -> 0, numTargetDeletionVectorsUpdated -> 0, scanTimeMs -> 0, numTargetRowsUpdated -> 0, numOutputRows -> 0, numTargetDeletionVectorsRemoved -> 0, numTargetRowsNotMatchedBySourceUpdated -> 0, numTargetChangeFilesAdded -> 0, numSourceRows -> 41510, numTargetFilesRemoved -> 0, numTargetRowsNotMatchedBySourceDeleted -> 0, rewriteTimeMs -> 1753)"
0,2026-10-28T14:30:11.000Z,WRITE,"Map(numFiles -> 1, numOutputRows -> 41510, numOutputBytes -> 1894236)"


In [0]:
# Compare valid Bronze events with Silver to identify differing columns.
# Read-only: does not modify Delta tables or checkpoints.

from pyspark.sql import functions as F
from pyspark.sql.window import Window

bronze_path = (
    "abfss://trackpulse-data@trackpulsestorage.dfs.core.windows.net/"
    "bronze/telemetry_stream/"
)

silver_path = (
    "abfss://trackpulse-data@trackpulsestorage.dfs.core.windows.net/"
    "silver/telemetry_stream/"
)

bronze = spark.read.format("delta").load(bronze_path)
silver = spark.read.format("delta").load(silver_path)

# Parse timestamps using the same logic as Cell 11.
prepared = (
    bronze
    .withColumn(
        "_event_ts",
        F.coalesce(
            F.to_timestamp("event_timestamp"),
            F.col("event_timestamp_utc")
        )
    )
    .filter(
        F.col("event_id").isNotNull()
        & (F.trim("event_id") != "")
        & F.col("_event_ts").isNotNull()
        & F.col("latitude").between(-90, 90)
        & F.col("longitude").between(-180, 180)
        & (F.col("speed_kph").isNull() | (F.col("speed_kph") >= 0))
        & (F.col("odometer_km").isNull() | (F.col("odometer_km") >= 0))
        & (F.col("rpm").isNull() | (F.col("rpm") >= 0))
        & (
            F.col("fuel_rate_lph").isNull()
            | (F.col("fuel_rate_lph") >= 0)
        )
    )
)

w = Window.partitionBy("event_id").orderBy(
    F.col("bronze_ingested_at").desc_nulls_last(),
    F.col("capture_sequence_number").desc_nulls_last()
)

source = (
    prepared
    .withColumn("_rn", F.row_number().over(w))
    .filter(F.col("_rn") == 1)
    .select(
        "event_id",
        F.col("_event_ts").alias("event_timestamp"),
        "vehicle_id", "latitude", "longitude", "speed_kph",
        "odometer_km", "rpm", "fuel_rate_lph", "engine_temp_c",
        "ignition_on", "trip_id", "shipment_id", "driver_id",
        "route_id", "cargo_temp_c",
        F.coalesce(
            F.col("bronze_ingested_at"),
            F.lit(None).cast("timestamp")
        ).alias("ingested_at"),
        F.lit(None).cast("string").alias("source_file")
    )
)

s = silver.select(
    "event_id",
    *[
        F.col(c).alias(f"silver_{c}")
        for c in silver.columns if c != "event_id"
    ]
)

joined = source.alias("b").join(
    s.alias("s"),
    F.col("b.event_id") == F.col("s.event_id"),
    "inner"
)

compare_cols = [
    "event_timestamp", "vehicle_id", "latitude", "longitude",
    "speed_kph", "odometer_km", "rpm", "fuel_rate_lph",
    "engine_temp_c", "ignition_on", "trip_id", "shipment_id",
    "driver_id", "route_id", "cargo_temp_c", "ingested_at",
    "source_file"
]

diff_exprs = [
    F.when(
        ~F.col(f"b.{c}").eqNullSafe(F.col(f"s.silver_{c}")),
        F.lit(c)
    )
    for c in compare_cols
]

diffs = joined.select(
    F.col("b.event_id").alias("event_id"),
    F.concat_ws(", ", *diff_exprs).alias("different_columns")
).filter(F.col("different_columns") != "")

print("Matched rows:", joined.count())
print("Rows with differences:", diffs.count())

display(
    diffs.groupBy("different_columns")
    .count()
    .orderBy(F.desc("count"))
)

Matched rows: 41510
Rows with differences: 0


different_columns,count


In [0]:
# Define the Silver columns used for change detection

silver_compare_columns = [
    "event_timestamp",
    "vehicle_id",
    "latitude",
    "longitude",
    "speed_kph",
    "odometer_km",
    "rpm",
    "fuel_rate_lph",
    "engine_temp_c",
    "ignition_on",
    "trip_id",
    "shipment_id",
    "driver_id",
    "route_id",
    "cargo_temp_c",
    "ingested_at",
    "source_file",
]

silver_changed_condition = "NOT (" + " AND ".join(
    f"t.`{c}` <=> s.`{c}`"
    for c in silver_compare_columns
) + ")"

print(silver_changed_condition)

NOT (t.`event_timestamp` <=> s.`event_timestamp` AND t.`vehicle_id` <=> s.`vehicle_id` AND t.`latitude` <=> s.`latitude` AND t.`longitude` <=> s.`longitude` AND t.`speed_kph` <=> s.`speed_kph` AND t.`odometer_km` <=> s.`odometer_km` AND t.`rpm` <=> s.`rpm` AND t.`fuel_rate_lph` <=> s.`fuel_rate_lph` AND t.`engine_temp_c` <=> s.`engine_temp_c` AND t.`ignition_on` <=> s.`ignition_on` AND t.`trip_id` <=> s.`trip_id` AND t.`shipment_id` <=> s.`shipment_id` AND t.`driver_id` <=> s.`driver_id` AND t.`route_id` <=> s.`route_id` AND t.`cargo_temp_c` <=> s.`cargo_temp_c` AND t.`ingested_at` <=> s.`ingested_at` AND t.`source_file` <=> s.`source_file`)


In [0]:
# Read the checkpoint metadata, offset 0, and commit 0
# This cell does not modify any files

checkpoint_path = (
    "abfss://trackpulse-data@trackpulsestorage.dfs.core.windows.net/"
    "checkpoints/databricks_streaming/silver_incremental_v1/"
)

files_to_read = [
    checkpoint_path + "metadata",
    checkpoint_path + "offsets/0",
    checkpoint_path + "commits/0",
]

for path in files_to_read:
    print(f"\n===== {path} =====")
    try:
        content = dbutils.fs.head(path, 10000)
        print(content)
    except Exception as e:
        print("Could not read file:", str(e))


===== abfss://trackpulse-data@trackpulsestorage.dfs.core.windows.net/checkpoints/databricks_streaming/silver_incremental_v1/metadata =====
{"id":"49074c21-fb9b-4efe-9065-81e4abc87d73"}

===== abfss://trackpulse-data@trackpulsestorage.dfs.core.windows.net/checkpoints/databricks_streaming/silver_incremental_v1/offsets/0 =====
v1
{"batchWatermarkMs":0,"batchTimestampMs":1793210576947,"conf":{"spark.sql.streaming.stateStore.providerClass":"com.databricks.sql.streaming.state.RocksDBStateStoreProvider","spark.databricks.streaming.dedup.preserveMetadataBoundary.enabled":"true","spark.sql.streaming.stateStore.rocksdb.mergeOperatorVersion":"1","spark.sql.streaming.stateStore.rocksdb.formatVersion":"5","spark.sql.streaming.statefulOperator.useStrictDistribution":"true","spark.sql.streaming.flatMapGroupsWithState.stateFormatVersion":"2","spark.sql.streaming.stateStore.rowChecksum.enabled":"false","spark.databricks.optimizer.joinOrToUnionAllExpansionWithEquiJoin.streaming":"false","spark.databric

In [0]:
# Inspect the checkpoint's recorded source offset and commit files
# Read-only: no checkpoint or Delta data is modified

checkpoint_path = (
    "abfss://trackpulse-data@trackpulsestorage.dfs.core.windows.net/"
    "checkpoints/databricks_streaming/silver_incremental_v1/"
)

print("Offset 0:")
print(dbutils.fs.head(checkpoint_path + "offsets/0", 20000))

print("\nCommit 0:")
print(dbutils.fs.head(checkpoint_path + "commits/0", 20000))

print("\nCheckpoint directory:")
for item in dbutils.fs.ls(checkpoint_path):
    print(item.name, item.size)

Offset 0:
v1
{"batchWatermarkMs":0,"batchTimestampMs":1793210576947,"conf":{"spark.sql.streaming.stateStore.providerClass":"com.databricks.sql.streaming.state.RocksDBStateStoreProvider","spark.databricks.streaming.dedup.preserveMetadataBoundary.enabled":"true","spark.sql.streaming.stateStore.rocksdb.mergeOperatorVersion":"1","spark.sql.streaming.stateStore.rocksdb.formatVersion":"5","spark.sql.streaming.statefulOperator.useStrictDistribution":"true","spark.sql.streaming.flatMapGroupsWithState.stateFormatVersion":"2","spark.sql.streaming.stateStore.rowChecksum.enabled":"false","spark.databricks.optimizer.joinOrToUnionAllExpansionWithEquiJoin.streaming":"false","spark.databricks.streaming.forEachBatch.optimized.aqeOverFrozenStatelessPlan.enabled":"false","spark.databricks.streaming.disableChangingStatelessShufflePartitionsForExistingQuery":"false","spark.sql.streaming.aggregation.stateFormatVersion":"2","spark.databricks.optimizer.convertCollectSetToCollectlistDistinct.enableStreaming":"

In [0]:
# Verify current Delta table versions and row counts before the retry test.
from delta.tables import DeltaTable

for name, path in [
    ("Bronze", bronze_path),
    ("Silver", silver_path),
    ("Quarantine", quarantine_path),
]:
    dt = DeltaTable.forPath(spark, path)
    print(f"\n{name} history:")
    dt.history(3).select(
        "version", "timestamp", "operation"
    ).show(truncate=False)


Bronze history:
+-------+-------------------+-----------------+
|version|timestamp          |operation        |
+-------+-------------------+-----------------+
|5      |2026-10-28 18:02:17|SET TBLPROPERTIES|
|4      |2026-10-28 17:56:19|SET TBLPROPERTIES|
|3      |2026-10-28 15:20:51|SET TBLPROPERTIES|
+-------+-------------------+-----------------+


Silver history:
+-------+-------------------+---------+
|version|timestamp          |operation|
+-------+-------------------+---------+
|2      |2026-10-28 18:08:01|MERGE    |
|1      |2026-10-28 18:03:03|MERGE    |
|0      |2026-10-28 14:30:11|WRITE    |
+-------+-------------------+---------+


Quarantine history:
+-------+-------------------+---------+
|version|timestamp          |operation|
+-------+-------------------+---------+
|5      |2026-10-28 18:18:58|UPDATE   |
|4      |2026-10-28 18:08:05|MERGE    |
|3      |2026-10-28 18:07:46|UPDATE   |
+-------+-------------------+---------+



In [0]:

# Capture a baseline before testing idempotency.
from delta.tables import DeltaTable

baseline = {
    "bronze_count": spark.read.format("delta").load(bronze_path).count(),
    "silver_count": spark.read.format("delta").load(silver_path).count(),
    "silver_distinct_ids": (
        spark.read.format("delta").load(silver_path)
        .select("event_id").distinct().count()
    ),
    "quarantine_count": (
        spark.read.format("delta").load(quarantine_path).count()
    ),
    "quarantine_distinct_keys": (
        spark.read.format("delta").load(quarantine_path)
        .select("quarantine_key").distinct().count()
    ),
}

print("Baseline before retry test:")
for key, value in baseline.items():
    print(f"{key}: {value}")


Baseline before retry test:
bronze_count: 41522
silver_count: 41510
silver_distinct_ids: 41510
quarantine_count: 12
quarantine_distinct_keys: 12


In [0]:

# Check for active streaming queries before retrying the pipeline.
for query in spark.streams.active:
    print(
        f"Query name: {query.name}, "
        f"query ID: {query.id}, "
        f"active: {query.isActive}"
    )

if len(spark.streams.active) == 0:
    print("No active streaming queries in this Spark session.")


No active streaming queries in this Spark session.


In [0]:

# Inspect the incremental checkpoint and the latest Delta table operations.
# This cell only reads metadata and does not modify any tables or checkpoints.

checkpoint_path = (
    "abfss://trackpulse-data@trackpulsestorage.dfs.core.windows.net/"
    "checkpoints/databricks_streaming/silver_incremental_v1/"
)

print("Checkpoint path:", checkpoint_path)

try:
    checkpoint_items = dbutils.fs.ls(checkpoint_path)
    print("Checkpoint exists and is accessible.")
    for item in checkpoint_items:
        print(f"{item.name} | directory={item.isDir()}")
except Exception as e:
    print("Checkpoint inspection failed:")
    print(str(e)[:1000])

# Review the most recent Silver and Quarantine operations.
from delta.tables import DeltaTable

for name, path in [
    ("Silver", silver_path),
    ("Quarantine", quarantine_path),
]:
    print(f"\n{name} latest history:")
    DeltaTable.forPath(spark, path).history(5).select(
        "version", "timestamp", "operation", "operationMetrics"
    ).show(truncate=False)


Checkpoint path: abfss://trackpulse-data@trackpulsestorage.dfs.core.windows.net/checkpoints/databricks_streaming/silver_incremental_v1/
Checkpoint exists and is accessible.
__tmp_path_dir/ | directory=True
commits/ | directory=True
metadata | directory=False
offsets/ | directory=True

Silver latest history:
+-------+-------------------+---------+----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

In [0]:

# Inspect streaming query status and available pipeline variables.
# This cell does not read or write any Delta table data.

print("Active streaming queries:", len(spark.streams.active))

for name in [
    "bronze_path",
    "silver_path",
    "quarantine_path",
    "silver_checkpoint_path",
]:
    print(f"{name} defined:", name in globals())


Active streaming queries: 0
bronze_path defined: True
silver_path defined: True
quarantine_path defined: True
silver_checkpoint_path defined: True


In [0]:
# Verify the latest events in Bronze without modifying data

bronze_path = (
    "abfss://trackpulse-data@trackpulsestorage.dfs.core.windows.net/"
    "bronze/telemetry_stream/"
)

bronze_df = spark.read.format("delta").load(bronze_path)

print("Total Bronze records:", bronze_df.count())

display(
    bronze_df
    .orderBy(
        F.col("bronze_ingested_at").desc_nulls_last()
    )
    .select(
        "event_id",
        "event_timestamp",
        "vehicle_id",
        "bronze_ingested_at"
    )
    .limit(10)
)